# DisVis in your browser

This notebook runs [DisVis](https://www.bonvinlab.org/disvis) entirely in your
browser, with [Pyodide](https://pyodide.org) and
[JupyterLite](https://jupyterlite.readthedocs.io).

Run all cells (*Run* > *Run All Cells*, or Shift+Enter through each one).
Loading Pyodide and DisVis takes 15-30 s the first time.

In [ ]:
import piplite

await piplite.install("disvis")

## Input

The PRE5-PUP2 test case of the DisVis repository: the fixed chain
(`O14250.pdb`), the scanning chain (`Q9UT97.pdb`) and 7 cross-links as distance
restraints (`restraints.dat`).

In [ ]:
from pyodide.http import pyfetch

base = "https://raw.githubusercontent.com/haddocking/disvis/main/test-cases/PRE5-PUP2-complex"
for name in ("O14250.pdb", "Q9UT97.pdb", "restraints.dat"):
    response = await pyfetch(f"{base}/{name}")
    with open(name, "wb") as f:
        f.write(await response.bytes())

print(open("restraints.dat").read())

## Run DisVis

A quick scan (15° rotational sampling, 2 Å grid), as the *Quick scanning* mode
of the web service; it takes about 30 s. The arguments are those of the
`disvis` command; the browser runs DisVis on a single core.

In [ ]:
%%time
import sys

from disvis import main

sys.argv = [
    "disvis", "O14250.pdb", "Q9UT97.pdb", "restraints.dat",
    "-a", "15", "-vs", "2", "--no-progress", "-d", "results",
]
main.main()

## Results

The number of complexes consistent with N restraints, and the z-score of each
restraint (the higher, the more likely a false positive).

In [ ]:
print(open("results/accessible_complexes.out").read())
print(open("results/z-score.out").read())

## Report

The report of the run, with the Mol* view of the accessible interaction space.
The files of the Mol* view are included in the page, as the notebook cannot
serve them. All output files are in the `results` folder of the file browser
(right-click a file to download it).

In [ ]:
import base64
import html
import json
from pathlib import Path

from IPython.display import HTML


def show_report(directory, height=800):
    """Show report.html, with the files of the Mol* view included."""
    d = Path(directory)
    state = json.loads((d / "state.mvsj").read_text())

    def include(node):
        url = node.get("params", {}).get("url")
        if url:
            data = base64.b64encode((d / url).read_bytes()).decode()
            node["params"]["url"] = "data:application/octet-stream;base64," + data
        for child in node.get("children", []):
            include(child)

    for snapshot in state["snapshots"]:
        include(snapshot["root"])
    page = (d / "report.html").read_text().replace(
        "mvsStories.loadFromURL('state.mvsj', { format: 'mvsj' });",
        f"mvsStories.loadFromData({json.dumps(json.dumps(state))}, {{ format: 'mvsj' }});",
    )
    return HTML(f'<div><iframe srcdoc="{html.escape(page)}" width="100%" height="{height}"></iframe></div>')


show_report("results")